In [1]:
import numpy as np
import pandas as pd
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
from bioframe.io.fileops import read_bigwig

In [2]:
import torch

In [3]:
from pyfaidx import Fasta

In [4]:
FASTA_FILE = "/project/fudenber_735/genomes/mm10/mm10.fa"
BED_FILE = "/project/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed"
COOL_FILE = "/project/fudenber_735/GEO/Hsieh2019/4DN/mESC_mm10_4DNFILZ1CPT8.mapq_30.2048.cool"
OUTPUT_DIR = "/scratch1/smaruj/train_pytorch_akita/mouse"
FOLD = 0

# --- Load Data ---
genome = Fasta(FASTA_FILE)
df = pd.read_csv(BED_FILE, sep="\t", header=None, names=["chrom", "start", "end", "fold"])
df_select_fold = df[df["fold"] == f"fold{FOLD}"].reset_index(drop=True)

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [5]:
import random

def one_hot_encode_sequence(sequence_obj):
    sequence = str(sequence_obj).upper()
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    encoded_sequence = np.array([
        base_to_int.get(base, base_to_int[random.choice("ACGT")]) for base in sequence
    ])

    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return np.expand_dims(one_hot_encoded, axis=0)


In [6]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=32, kernel_stddev=1):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
        
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
    
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns

In [8]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [11]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector


In [14]:
for i, row in enumerate(df_select_fold[-5:].itertuples(index=False)):
    chrom, start, end = row.chrom, row.start, row.end
    mseq_str = f"{chrom}:{start}-{end}"

    print(f"Seq {i} | Coordinates:", mseq_str)
    
    sequence = genome[chrom][start:end]
    ohe_sequence = one_hot_encode_sequence(sequence)
    # matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=32, sigma=1, width=5)
    
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1)
    
    vector = upper_triangular_to_vector_skip_diagonals(matrix)
    
    print(matrix.shape)
    print(vector.shape)
    
    # if np.any(np.isnan(matrix)):

    #     print("min=", np.nanmin(matrix))
    #     print("max=", np.nanmax(matrix))
        
    # plot_map(matrix, vmin=-0.6, vmax=0.6)
    
    # seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    # seq_hic_nan = np.isnan(seq_hic_raw)
    # num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    # print("num_filtered_bins:", num_filtered_bins)
    
    # # set blacklist to NaNs
    
    # # print("min=", np.nanmin(seq_hic_raw))
    # # print("max=", np.nanmax(seq_hic_raw))
    
    # # clip first diagonals and high values
    # clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    # print("clipval=", clipval)
    # for i in range(-diagonal_offset+1, diagonal_offset):
    #     set_diag(seq_hic_raw, clipval, i)
    # seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    # seq_hic_raw[seq_hic_nan] = np.nan
    
    # # print("min=", np.nanmin(seq_hic_raw))
    # # print("max=", np.nanmax(seq_hic_raw))
    
    # # adaptively coarsegrain based on raw counts
    # seq_hic_smoothed = adaptive_coarsegrain(
    #                         seq_hic_raw,
    #                         genome_hic_cool.matrix(balance=False).fetch(mseq_str),
    #                         cutoff=2, max_levels=8)
    # seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # # print("min=", np.nanmin(seq_hic_raw))
    # # print("max=", np.nanmax(seq_hic_raw))
    
    # # local obs/exp
    # seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    # seq_hic_obsexp = np.log(seq_hic_obsexp)
    
    # seq_hic_obsexp = interp_nan(seq_hic_obsexp)
    # for i in range(-diagonal_offset+1, diagonal_offset): set_diag(seq_hic_obsexp, 0,i)
    
    # kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    # seq_hic = convolve(seq_hic_obsexp, kernel)
    
    # if np.any(np.isnan(seq_hic)):
    
    #     print("min=", np.nanmin(seq_hic))
    #     print("max=", np.nanmax(seq_hic))
        
    #     plot_map(seq_hic, vmin=-0.6, vmax=0.6)
    
    # plot_matrix(seq_hic_raw, cmap=fruitpunch, title="Balance = True", xlabel="X-Axis", ylabel="Y-Axis")
    # plot_matrix(matrix, cmap="RdBu_r", title="Sample Matrix", xlabel="X-Axis", ylabel="Y-Axis")

    

Seq 0 | Coordinates: chr5:150165504-151476224
num_filtered_bins: 11
(512, 512)
(130305,)
Seq 1 | Coordinates: chr11:66576384-67887104
num_filtered_bins: 9
(512, 512)
(130305,)
Seq 2 | Coordinates: chrX:57774080-59084800
num_filtered_bins: 24
(512, 512)
(130305,)
Seq 3 | Coordinates: chr11:67559424-68870144
num_filtered_bins: 3
(512, 512)
(130305,)
Seq 4 | Coordinates: chr3:100540416-101851136
num_filtered_bins: 0
(512, 512)
(130305,)


In [ ]:
def generate_dataset(df, genome, genome_hic_cool):
    """
    Iterates through genomic regions, extracts DNA sequences and Hi-C matrices,
    and saves the data in a PyTorch-compatible format.

    Args:
        df (pd.DataFrame): DataFrame containing genomic regions
        genome (pyfaidx.Fasta): FASTA genome object
        genome_hic_cool (cooler.Cooler): Cooler object for Hi-C data

    Returns:
        list: List of (ohe_sequence, hic_matrix) tuples
    """
    data_list = []
    
    for row in df.itertuples(index=False):
        chrom, start, end = row.chrom, row.start, row.end
        mseq_str = f"{chrom}:{start}-{end}"
        
        # logging.info(f"Processing {mseq_str}")
        
        # One-hot encode sequence
        sequence = genome[chrom][start:end]
        ohe_sequence = one_hot_encode_sequence(sequence)

        # Process Hi-C matrix
        matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=32, sigma=1, width=5)

        # Convert to PyTorch tensors
        ohe_tensor = torch.tensor(ohe_sequence, dtype=torch.float32)  # Shape: (1, 4, seq_length)
        matrix_tensor = torch.tensor(matrix, dtype=torch.float32)  # Shape: (matrix_size, matrix_size)

        data_list.append((ohe_tensor, matrix_tensor))
    
    return data_list


In [ ]:
dataset = generate_dataset(df_train_fold[:5], genome, genome_hic_cool)

In [ ]:
for row in df_train_fold.itertuples(index=False):
    chrom, start, end = row.chrom, row.start, row.end - 262144   # difference between Akita.V1 and V2 window
    print(f"Processing {chrom}: {start}-{end}")
    
    mseq_str = '%s:%d-%d' % (chrom, start, end)
    
    sequence = genome[chrom][start:end]
    ohe_sequence = one_hot_encode_sequence(sequence)
    print(ohe_sequence.shape)
    
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=32, sigma=1, width=5)
    
    # Plot the matrix
    plt.figure(figsize=(8, 8))
    plt.matshow(matrix.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
    plt.colorbar()
    plt.title('Heatmap of the Matrix')
    plt.show()
    

In [ ]:
import matplotlib.pyplot as plt

# Plot the matrix
plt.figure(figsize=(8, 8))
plt.matshow(kernel_log_hic_obsexp.astype(np.float16), cmap='RdBu_r', vmin=-0.6, vmax=0.6)
plt.colorbar()
plt.title('Heatmap of the Matrix')
plt.show()